# LoRA reranker fine-tuning on Colab GPU

Runs `lora_reranker.py` against the FinAgentBench data on a Colab T4 GPU,
writes the LoRA adapter and `metrics.json` to the existing
`gs://ftrag-adapters-ml-cloud-hw2-250147` bucket so the rest of the cloud
pipeline (offline merge, vLLM serving) can consume it unchanged.

## Before you start
1. Runtime -> Change runtime type -> Hardware accelerator: T4 GPU
2. Run cells top to bottom. Cell 2 will pop a Google login — use the
   same `ts3479@columbia.edu` account that owns the GCS buckets.

Total wall time on a T4: ~30-50 min.

In [ ]:
PROJECT_ID = 'ml-cloud-hw2-250147'
DATA_BUCKET = 'ftrag-data-ml-cloud-hw2-250147'
ADAPTERS_BUCKET = 'ftrag-adapters-ml-cloud-hw2-250147'
RUN_NAME = 'llama3_lora_reranker_colab'

import subprocess, sys, os
print('Python:', sys.version.split()[0])
print('GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

In [ ]:
from google.colab import auth
auth.authenticate_user()
subprocess.run(['gcloud', 'config', 'set', 'project', PROJECT_ID], check=True)
print('GCS auth OK')

In [ ]:
%pip install -q --upgrade pip
%pip install -q transformers==4.46.3 peft==0.14.0 accelerate==1.2.1 huggingface_hub==0.27.1 safetensors tqdm sentencepiece
print('deps installed')

In [ ]:
import os
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/code', exist_ok=True)
os.makedirs('/content/outputs', exist_ok=True)

# Pull the 4 FinAgentBench JSONL files from GCS
subprocess.run(['gcloud', 'storage', 'cp',
  f'gs://{DATA_BUCKET}/document_ranking_kaggle_dev.jsonl',
  f'gs://{DATA_BUCKET}/chunk_ranking_kaggle_dev.jsonl',
  f'gs://{DATA_BUCKET}/document_ranking_kaggle_eval.jsonl',
  f'gs://{DATA_BUCKET}/chunk_ranking_kaggle_eval.jsonl',
  '/content/data/'], check=True)

subprocess.run(['ls', '-lh', '/content/data/'], check=True)

In [ ]:
# Pull the lora_reranker.py script from the ftRAG GitHub repo (or paste it inline)
import urllib.request
url = 'https://raw.githubusercontent.com/yanhao13/ftRAG/main/data/lora_reranker.py'
urllib.request.urlretrieve(url, '/content/code/lora_reranker.py')
print('lora_reranker.py downloaded:', os.path.getsize('/content/code/lora_reranker.py'), 'bytes')

In [ ]:
# === FULL TRAINING RUN ===
# Trains LoRA on FinAgentBench dev rows and evaluates on the held-out validation
# slice (500 chunk queries, 40 doc queries) to produce comparable numbers.
#
# Note: chunk-val-queries=500 means we're scoring on the SAME 500-query split as
# chunk_best_ensemble.py uses, so the LoRA's standalone nDCG@5 number from this
# run is directly comparable to the supervised-ensemble 0.432.

import time
t0 = time.time()

result = subprocess.run([
    'python3', '/content/code/lora_reranker.py',
    '--doc-dev=/content/data/document_ranking_kaggle_dev.jsonl',
    '--chunk-dev=/content/data/chunk_ranking_kaggle_dev.jsonl',
    '--doc-eval=/content/data/document_ranking_kaggle_eval.jsonl',
    '--chunk-eval=/content/data/chunk_ranking_kaggle_eval.jsonl',
    f'--output-dir=/content/outputs/{RUN_NAME}',
    '--epochs=1',
    '--batch-size=2',
    '--gradient-accumulation-steps=8',
    '--doc-query-limit=200',
    '--chunk-query-limit=200',
    '--doc-val-queries=40',
    '--chunk-val-queries=500',
    '--max-train-pairs=2000',
    '--max-length=192',
])

print(f'\nTraining + eval finished in {(time.time()-t0)/60:.1f} min, exit={result.returncode}')

In [ ]:
import json
metrics_path = f'/content/outputs/{RUN_NAME}/metrics.json'
with open(metrics_path) as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2, sort_keys=True))

In [ ]:
# Push everything back to the existing GCS bucket so the rest of the cloud
# pipeline (merge, serve) can pick it up unchanged.
subprocess.run(['gcloud', 'storage', 'cp', '-r',
    f'/content/outputs/{RUN_NAME}',
    f'gs://{ADAPTERS_BUCKET}/'], check=True)

# Verify
subprocess.run(['gcloud', 'storage', 'ls', '-r',
    f'gs://{ADAPTERS_BUCKET}/{RUN_NAME}/'], check=True)

## Done

The LoRA adapter and `metrics.json` are now in
`gs://ftrag-adapters-ml-cloud-hw2-250147/llama3_lora_reranker_colab/`.

Tell the agent on your Mac "colab run finished" and it will:
1. Pull the metrics into `docs/cloud/cloud_evidence/`
2. Update the architecture diagram to reflect Colab as the training compute
3. Run the offline merge step (CPU only, doesn't need GPU)